# ECG Arrhythmia Classification — Live Demo

**Model:** CNN-Transformer (tuned + hyperparameter-optimized + lead-masked)
**Dataset:** CPSC 2018 — 12-lead ECG, 9 arrhythmia classes
**Goal of this notebook:** demonstrate how the trained model performs on real test samples.

The notebook does NOT train anything — it loads a pre-trained checkpoint and runs forward passes.


## 1. Setup — load the model and test data

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # CPU inference (deterministic, demo-friendly)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import tensorflow as tf
import pickle
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

from data_processing.data_generator import DataGenerator
from explainability.lead_importance import (
    build_tuned_cnn_transformer, LEAD_NAMES, CHECKPOINT_PATH,
)
from utils.utils import input_length, num_channels, num_classes
from config import SAMPLING_RATE

print(f"TensorFlow {tf.__version__}")
print(f"Loading checkpoint: {CHECKPOINT_PATH}")

In [ ]:
model = build_tuned_cnn_transformer((input_length, num_channels), num_classes)
model.load_weights(CHECKPOINT_PATH)
print(f"Model loaded — {model.count_params():,} parameters")

In [ ]:
with open("data/test_data.pkl", "rb") as f:
    test_data = pickle.load(f)

label_encoder = LabelEncoder()
label_encoder.fit(test_data["classes"])
class_names = list(label_encoder.classes_)

test_gen = DataGenerator(test_data, label_encoder, shuffle=False, augment=False)
all_signals, all_labels = [], []
for x, y in test_gen:
    all_signals.append(x); all_labels.append(y)
all_signals = np.concatenate(all_signals)
all_labels = np.concatenate(all_labels)

print(f"Loaded {len(all_signals)} test samples")
print(f"Classes: {class_names}")

## 2. Live demo — predictions on individual test samples

For each chosen sample we:
1. Plot the ECG signal (lead II)
2. Run a single forward pass through the trained model
3. Show the predicted class and confidence (softmax) for all 9 classes

Green bar = true class, red bar = predicted class (when wrong), gray = others.

In [ ]:
def demo_sample(signal, true_idx, sample_id):
    """Plot ECG + softmax confidence for one test sample."""
    probs = model.predict(signal[np.newaxis], verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    correct = (pred_idx == true_idx)

    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(15, 4),
        gridspec_kw={"width_ratios": [3, 1]},
    )

    time = np.arange(signal.shape[0]) / SAMPLING_RATE
    ax1.plot(time, signal[:, 1], color="black", linewidth=0.6, label="Lead II")
    ax1.set_xlabel("Time (s)"); ax1.set_ylabel("Amplitude (mV)")
    title_color = "darkgreen" if correct else "darkred"
    ax1.set_title(
        f"Sample #{sample_id} — True: {class_names[true_idx]} | "
        f"Predicted: {class_names[pred_idx]} ({probs[pred_idx]:.1%})",
        fontsize=11, color=title_color,
    )
    ax1.grid(alpha=0.3)
    ax1.legend(loc="upper right", fontsize=8)

    colors = ["green" if i == true_idx
              else ("red" if i == pred_idx and not correct else "lightgray")
              for i in range(len(class_names))]
    ax2.barh(class_names, probs, color=colors, edgecolor="black", linewidth=0.5)
    ax2.set_xlim(0, 1); ax2.set_xlabel("Probability")
    ax2.set_title("Class confidence")
    ax2.invert_yaxis()
    for i, v in enumerate(probs):
        if v > 0.04:
            ax2.text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=8)

    plt.tight_layout(); plt.show()

    status = "CORRECT" if correct else "INCORRECT"
    print(f"   {status} -> predicted '{class_names[pred_idx]}' "
          f"with {probs[pred_idx]:.1%} confidence (true label: '{class_names[true_idx]}')")

In [ ]:
# Pick one representative sample for each class we want to showcase
demo_classes = ["AF", "RBBB", "SNR", "PVC", "STE"]
demo_indices = {}
for c in demo_classes:
    cls_idx = class_names.index(c)
    sample_idx = int(np.where(all_labels == cls_idx)[0][0])
    demo_indices[c] = sample_idx
    print(f"{c:5s} -> sample index {sample_idx}")

### Sample 1 — Atrial Fibrillation (AF)

In [ ]:
idx = demo_indices["AF"]
demo_sample(all_signals[idx], all_labels[idx], idx)

### Sample 2 — Right Bundle-Branch Block (RBBB)

In [ ]:
idx = demo_indices["RBBB"]
demo_sample(all_signals[idx], all_labels[idx], idx)

### Sample 3 — Sinus Rhythm (SNR)

In [ ]:
idx = demo_indices["SNR"]
demo_sample(all_signals[idx], all_labels[idx], idx)

### Sample 4 — Premature Ventricular Contraction (PVC)

In [ ]:
idx = demo_indices["PVC"]
demo_sample(all_signals[idx], all_labels[idx], idx)

### Sample 5 — ST Elevation (STE) — the rarest class in the dataset

In [ ]:
idx = demo_indices["STE"]
demo_sample(all_signals[idx], all_labels[idx], idx)

## 3. Explainability — Grad-CAM (which time regions matter)

Grad-CAM highlights which timesteps in the ECG drove the predicted class.
Red = high importance, green = low.

In [ ]:
GRADCAM_LAYER = "gradcam_target"
grad_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=[model.get_layer(GRADCAM_LAYER).output, model.output],
)

def compute_gradcam(signal, class_idx):
    x = tf.cast(signal[np.newaxis], tf.float32)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(x, training=False)
        score = preds[:, class_idx]
    grads = tape.gradient(score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

idx = demo_indices["AF"]
signal = all_signals[idx]; true_idx = all_labels[idx]
hm = compute_gradcam(signal, true_idx)

time = np.arange(signal.shape[0]) / SAMPLING_RATE
hm_up = np.interp(np.linspace(0, len(hm) - 1, len(time)),
                  np.arange(len(hm)), hm)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5),
                               gridspec_kw={"height_ratios": [4, 1]},
                               sharex=True)
ax1.plot(time, signal[:, 1], color="black", linewidth=0.6)
ax1.set_ylabel("Amplitude (mV)")
ax1.set_title(f"Grad-CAM heatmap on AF sample — model focused on these regions to predict '{class_names[true_idx]}'")

cmap = plt.get_cmap("RdYlGn_r")
for i in range(len(time) - 1):
    ax2.axvspan(time[i], time[i + 1], color=cmap(hm_up[i]), alpha=0.85)
ax2.set_yticks([]); ax2.set_xlabel("Time (s)")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1)); sm.set_array([])
plt.colorbar(sm, ax=ax2, orientation="horizontal", pad=0.4, fraction=0.6,
             label="Importance (red = high, green = low)")
plt.tight_layout(); plt.show()

## 4. Explainability — per-lead importance for one sample

Input-gradient saliency tells us which of the 12 ECG leads contributed most to this specific prediction.

In [ ]:
def lead_saliency(signal, class_idx):
    x = tf.convert_to_tensor(signal[np.newaxis], dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x)
        score = model(x, training=False)[:, class_idx]
    grad = tape.gradient(score, x)[0].numpy()
    contribution = np.abs(grad) * np.abs(signal)
    return contribution.sum(axis=0)

idx = demo_indices["RBBB"]
signal = all_signals[idx]; true_idx = all_labels[idx]
sal = lead_saliency(signal, true_idx)
sal_norm = sal / sal.max()

fig, ax = plt.subplots(figsize=(10, 4))
order = np.argsort(sal_norm)[::-1]
colors = ["#C44E52" if i < 3 else "#4C72B0" for i in range(12)]
ax.bar([LEAD_NAMES[i] for i in order],
       [sal_norm[i] for i in order], color=colors)
ax.set_ylabel("Normalized saliency"); ax.set_xlabel("ECG lead")
ax.set_title(f"Per-lead importance for this RBBB sample — top 3 in red")
plt.tight_layout(); plt.show()

print("Top 3 leads for this sample:", [LEAD_NAMES[i] for i in order[:3]])

## 5. Wearable scenario — same sample, only V1/II/V5 active

We zero out 9 of the 12 leads, keeping only the clinical Holter montage. The model was trained
with random lead masking, so it handles this gracefully.

In [ ]:
HOLTER = ["V1", "II", "V5"]
keep_idx = [LEAD_NAMES.index(l) for l in HOLTER]

idx = demo_indices["RBBB"]
signal = all_signals[idx]; true_idx = all_labels[idx]

masked = np.zeros_like(signal)
masked[:, keep_idx] = signal[:, keep_idx]

probs_full = model.predict(signal[np.newaxis], verbose=0)[0]
probs_3lead = model.predict(masked[np.newaxis], verbose=0)[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(class_names)); w = 0.4
ax1.bar(x - w/2, probs_full, w, label="12-lead (full)", color="#4C72B0")
ax1.bar(x + w/2, probs_3lead, w, label="3-lead V1/II/V5", color="#DD8452")
ax1.axvline(true_idx, linestyle="--", color="green", alpha=0.7, label=f"True: {class_names[true_idx]}")
ax1.set_xticks(x); ax1.set_xticklabels(class_names)
ax1.set_ylabel("Probability"); ax1.set_ylim(0, 1)
ax1.set_title("Class confidence: 12-lead vs Holter 3-lead")
ax1.legend(fontsize=9)

time = np.arange(signal.shape[0]) / SAMPLING_RATE
for i, name in enumerate(HOLTER):
    li = LEAD_NAMES.index(name)
    ax2.plot(time, signal[:, li] + i * 3, label=name, linewidth=0.6)
ax2.set_xlabel("Time (s)"); ax2.set_yticks([])
ax2.set_title("The 3 active leads (V1, II, V5)")
ax2.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f"12-lead prediction: {class_names[probs_full.argmax()]} ({probs_full.max():.1%})")
print(f"3-lead prediction:  {class_names[probs_3lead.argmax()]} ({probs_3lead.max():.1%})")
print(f"True label:         {class_names[true_idx]}")

## Summary

- The model performs single-pass inference on a 15000-timestep, 12-lead ECG window.
- Predicted class and per-class confidence are produced in milliseconds.
- Explainability (Grad-CAM, lead saliency) shows what the model relies on.
- Despite being a 12-lead model, it remains usable on a 3-lead wearable montage thanks to lead-masking augmentation during training.

For the full quantitative analysis (per-class F1, multi-montage comparison, model variants),
see the scripts in `explainability/` — they reproduce the numbers used in the slides.